In [21]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib
from tqdm import tqdm
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, recall_score, precision_score, f1_score
from skimage.transform import resize

In [4]:
DATASET_PATH = "ct-ich"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
scan_dir = os.path.join(DATASET_PATH, "ct_scans")
mask_dir = os.path.join(DATASET_PATH, "masks")


In [5]:
files = sorted(os.listdir(scan_dir))

In [8]:
X = []
y = []

for file in tqdm(files):

    scan = nib.load(os.path.join(scan_dir, file)).get_fdata()
    mask = nib.load(os.path.join(mask_dir, file)).get_fdata()

    for i in range(scan.shape[2]):

        img = scan[:,:,i]
        m = mask[:,:,i]

        label = int(m.sum() > 0)

        X.append(img)
        y.append(label)

100%|██████████████████████████████████████████████████████████████████████████████████| 75/75 [00:09<00:00,  7.66it/s]


In [9]:
print("Total slices:", len(X))
print("Positive slices:", sum(y))

Total slices: 2814
Positive slices: 318


In [10]:
processed = []

for img in tqdm(X):

    img = np.clip(img, -40, 80)
    img = (img + 40) / 120

    img = resize(img, (256,256))

    processed.append(img)

100%|█████████████████████████████████████████████████████████████████████████████| 2814/2814 [00:15<00:00, 184.52it/s]


In [11]:
X = np.array(processed)
y = np.array(y)

print(X.shape)

(2814, 256, 256)


In [12]:
X = np.expand_dims(X, axis=1)

print(X.shape)

(2814, 1, 256, 256)


In [13]:
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)

In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [17]:
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

In [18]:
import torch.nn as nn
import torch.nn.functional as F

class HemorrhageCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)

        self.pool = nn.MaxPool2d(2)

        self.fc1 = nn.Linear(64 * 32 * 32, 128)
        self.fc2 = nn.Linear(128, 2)

    def forward(self, x):

        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))

        x = x.view(x.size(0), -1)

        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x

In [19]:
model = HemorrhageCNN().to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [20]:
for epoch in range(10):

    model.train()
    total_loss = 0

    for x_batch, y_batch in train_loader:

        x_batch = x_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(x_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print("Epoch:", epoch+1, "Loss:", total_loss/len(train_loader))

Epoch: 1 Loss: 0.33059907745983863
Epoch: 2 Loss: 0.23569668870833185
Epoch: 3 Loss: 0.1790431203941504
Epoch: 4 Loss: 0.13844988577895695
Epoch: 5 Loss: 0.1167686606446902
Epoch: 6 Loss: 0.08067203452810645
Epoch: 7 Loss: 0.0655095189706319
Epoch: 8 Loss: 0.05210802090975145
Epoch: 9 Loss: 0.03126575740316184
Epoch: 10 Loss: 0.03850969582951317


In [22]:
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():

    for x_batch, y_batch in test_loader:

        x_batch = x_batch.to(DEVICE)

        outputs = model(x_batch)
        preds = torch.argmax(outputs, dim=1).cpu()

        all_preds.extend(preds.numpy())
        all_labels.extend(y_batch.numpy())

In [23]:
print(confusion_matrix(all_labels, all_preds))
print(classification_report(all_labels, all_preds))

[[496   3]
 [ 17  47]]
              precision    recall  f1-score   support

           0       0.97      0.99      0.98       499
           1       0.94      0.73      0.82        64

    accuracy                           0.96       563
   macro avg       0.95      0.86      0.90       563
weighted avg       0.96      0.96      0.96       563



In [24]:
import os
import json
import numpy as np
import torch
import matplotlib.pyplot as plt
from datetime import datetime

OUTPUT_DIR = "model2_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = os.path.join(OUTPUT_DIR, RUN_ID)
os.makedirs(RUN_DIR, exist_ok=True)

print("Saving outputs to:", RUN_DIR)

Saving outputs to: model2_output\20260315_184202


In [27]:
np.save(os.path.join(RUN_DIR, "X_train_shape.npy"), np.array(X_train.shape))
np.save(os.path.join(RUN_DIR, "X_test_shape.npy"), np.array(X_test.shape))

torch.save({
    "X_train": X_train,
    "y_train": y_train,
    "X_test": X_test,
    "y_test": y_test
}, os.path.join(RUN_DIR, "dataset_tensors.pt"))

In [29]:
from sklearn.metrics import classification_report, confusion_matrix

cm = confusion_matrix(all_labels, all_preds)
report = classification_report(all_labels, all_preds, output_dict=True)

np.save(os.path.join(RUN_DIR, "confusion_matrix.npy"), cm)

with open(os.path.join(RUN_DIR, "classification_report.json"), "w") as f:
    json.dump(report, f, indent=4)

In [30]:
torch.save(model.state_dict(), os.path.join(RUN_DIR, "model_weights.pt"))
torch.save(model, os.path.join(RUN_DIR, "full_model.pt"))

In [33]:
#testing on Qure.ai HeadCT

import os, re
import torch
import numpy as np
import pandas as pd
import pydicom
from tqdm import tqdm
from skimage.transform import resize
from sklearn.metrics import confusion_matrix, classification_report

dataset_root = "crawford/qureai-headct/versions/2"

# ---- LOAD LABELS ----
labels_df = pd.read_csv(os.path.join(dataset_root, "reads.csv"))

label_dict = {}
for _, row in labels_df.iterrows():
    num = re.findall(r'\d+', row["name"])[-1]
    label_dict[num] = int(row["R1:ICH"])

# ---- GET PATIENT PATHS ----
patient_paths = []
for batch in os.listdir(dataset_root):
    batch_path = os.path.join(dataset_root, batch)
    if not os.path.isdir(batch_path):
        continue
    for patient in os.listdir(batch_path):
        if patient.startswith("."):
            continue
        patient_paths.append(os.path.join(batch_path, patient))

# ---- FIND CT SERIES ----
def find_series(patient_path):
    for root, dirs, files in os.walk(patient_path):
        if len(files) > 10 and files[0].endswith(".dcm"):
            if "ct plain thin" in root.lower():
                return root
    return None

# ---- LOAD DICOM SLICES ----
def load_slices(series_path):
    slices = []
    for f in os.listdir(series_path):
        if f.endswith(".dcm"):
            dcm = pydicom.dcmread(os.path.join(series_path, f))
            slices.append(dcm)
    slices.sort(key=lambda x: float(x.ImagePositionPatient[2]))
    return slices

# ---- DICOM → HU ----
def dicom_to_hu(dcm):
    img = dcm.pixel_array.astype(np.float32)
    return img * dcm.RescaleSlope + dcm.RescaleIntercept

# ---- MODEL SLICE INFERENCE ----
def predict_slice(img):

    img = np.clip(img, -40, 80)
    img = (img + 40) / 120
    img = resize(img, (256,256))

    img = torch.tensor(img, dtype=torch.float32).unsqueeze(0).unsqueeze(0)

    with torch.no_grad():
        out = model(img)

    prob = torch.softmax(out, dim=1)[0,1].item()

    return prob

# ---- SCAN PREDICTION ----
def predict_scan(slices):

    probs = []

    for s in slices:
        img = dicom_to_hu(s)
        p = predict_slice(img)
        probs.append(p)

    return max(probs)

# ---- EVALUATION ----
y_true = []
y_pred = []
for patient in tqdm(patient_paths):
    pid = re.findall(r'\d+', os.path.basename(patient))[-1]

    if pid not in label_dict:
        continue

    series = find_series(patient)
    if series is None:
        continue

    slices = load_slices(series)

    prob = predict_scan(slices)

    pred = int(prob > 0.5)
    label = label_dict[pid]

    y_pred.append(pred)
    y_true.append(label)

print("Evaluated scans:", len(y_true))
print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred))

# Evaluated scans: 100
# [[21 34]
#  [10 35]]
#               precision    recall  f1-score   support

#            0       0.68      0.38      0.49        55
#            1       0.51      0.78      0.61        45

#     accuracy                           0.56       100
#    macro avg       0.59      0.58      0.55       100
# weighted avg       0.60      0.56      0.54       100

100%|████████████████████████████████████████████████████████████████████████████████| 350/350 [10:57<00:00,  1.88s/it]

Evaluated scans: 119
[[25 40]
 [10 44]]
              precision    recall  f1-score   support

           0       0.71      0.38      0.50        65
           1       0.52      0.81      0.64        54

    accuracy                           0.58       119
   macro avg       0.62      0.60      0.57       119
weighted avg       0.63      0.58      0.56       119

